
# Project 15 — Colab Notebook for Stability Benchmarking with ASR / FSR / Screening List

This notebook is designed to be **Google Colab ready** and to run end-to-end on the uploaded CSV files:

- `ASR_data_SI_20250204.csv`
- `FSR_data_SI_20250204.csv`
- `ION_data_SI_20250204.csv`
- `ASR_FSR_check.csv`
- `12089-recommended-screening-list.csv`

## What this notebook does

1. Ingests and audits all input tables  
2. Builds normalized clean tables for ASR and FSR  
3. Engineers descriptor regimes A–D  
4. Creates matched ASR–FSR paired datasets  
5. Runs regression and thresholded screening analyses  
6. Evaluates random, grouped, and transfer-validation settings  
7. Performs chemistry-facing subgroup and failure analyses  
8. Produces publication-oriented main figures and SI figures  
9. Saves all intermediate tables, model results, and log files  
10. Supports **resume from checkpoint** if outputs already exist

## Design principles

- clean and explicit data handling
- extensive logging
- intermediate saves after major stages
- rerunnable and resumable
- lightweight, interpretable ML rather than heavy black-box pipelines
- figures saved in publication-friendly high resolution


In [ ]:

# ============================================================
# 0. Optional: mount Google Drive (recommended in Colab)
# ============================================================
# Uncomment if you want all outputs saved directly to Google Drive.
# from google.colab import drive
# drive.mount('/content/drive')

# Example:
# PROJECT_ROOT = "/content/drive/MyDrive/project15_stability_benchmark"
# If not mounted, the notebook will use the current working directory.


In [ ]:

# ============================================================
# 1. Imports and global configuration
# ============================================================
import os
import re
import json
import time
import math
import shutil
import pickle
import warnings
import traceback
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from scipy.stats import spearmanr

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.metrics import (
    r2_score, mean_absolute_error, mean_squared_error,
    roc_auc_score, average_precision_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score
)
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
from sklearn.model_selection import (
    train_test_split, GroupShuffleSplit, GroupKFold
)

warnings.filterwarnings("ignore")

# -----------------------------
# User configuration
# -----------------------------
PROJECT_ROOT = Path(os.environ.get("PROJECT15_ROOT", ".")).resolve()

RAW_DIR = PROJECT_ROOT / "raw_data"
OUT_DIR = PROJECT_ROOT / "project15_outputs"
CHECKPOINT_DIR = OUT_DIR / "checkpoints"
TABLE_DIR = OUT_DIR / "tables"
FIG_DIR = OUT_DIR / "figures"
LOG_DIR = OUT_DIR / "logs"
MODEL_DIR = OUT_DIR / "models"

for d in [RAW_DIR, OUT_DIR, CHECKPOINT_DIR, TABLE_DIR, FIG_DIR, LOG_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# File names expected either in PROJECT_ROOT or RAW_DIR
FILE_MAP = {
    "ASR": "ASR_data_SI_20250204.csv",
    "FSR": "FSR_data_SI_20250204.csv",
    "ION": "ION_data_SI_20250204.csv",
    "CROSSWALK": "ASR_FSR_check.csv",
    "SCREENING": "12089-recommended-screening-list.csv",
}

RANDOM_STATE = 42
N_RANDOM_REPEATS = 5
TOP_K_IMPORTANCE = 20
TOP_CONFIDENCE_FRACTION = 0.10

TARGETS = [
    "Solvent_stability",
    "Water_stability",
    "Thermal_stability (℃)",
]

PRIMARY_REGRESSION_TARGETS = [
    "Solvent_stability",
    "Water_stability",
]
SUPPLEMENTARY_REGRESSION_TARGETS = [
    "Thermal_stability (℃)",
]

CLASSIFICATION_CUTOFFS = [0.60, 0.70, 0.80]

CHECKPOINT_POLICY = {
    "load_if_exists": True,   # load stage checkpoint if available
    "save_after_stage": True, # save stage outputs
    "force_rerun": set(),     # e.g. {"stage_04_features"} to recompute a stage
}

print("PROJECT_ROOT =", PROJECT_ROOT)
print("OUT_DIR      =", OUT_DIR)


In [ ]:

# ============================================================
# 2. Logging and checkpoint utilities
# ============================================================
LOG_FILE = LOG_DIR / f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

def log(msg, also_print=True):
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{ts}] {msg}"
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(line + "\n")
    if also_print:
        print(line)

def save_pickle(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "wb") as f:
        pickle.dump(obj, f)

def load_pickle(path):
    with open(path, "rb") as f:
        return pickle.load(f)

def stage_checkpoint_path(stage_name):
    return CHECKPOINT_DIR / f"{stage_name}.pkl"

def maybe_load_stage(stage_name):
    path = stage_checkpoint_path(stage_name)
    if (
        CHECKPOINT_POLICY["load_if_exists"]
        and path.exists()
        and stage_name not in CHECKPOINT_POLICY["force_rerun"]
    ):
        log(f"Loading checkpoint for {stage_name}: {path}")
        return load_pickle(path)
    return None

def save_stage(stage_name, obj):
    if CHECKPOINT_POLICY["save_after_stage"]:
        path = stage_checkpoint_path(stage_name)
        save_pickle(obj, path)
        log(f"Saved checkpoint for {stage_name}: {path}")

def export_df(df, name, index=False):
    csv_path = TABLE_DIR / f"{name}.csv"
    pkl_path = TABLE_DIR / f"{name}.pkl"
    df.to_csv(csv_path, index=index)
    df.to_pickle(pkl_path)
    log(f"Exported table: {csv_path.name} and {pkl_path.name}")

def safe_numeric(series):
    return pd.to_numeric(series, errors="coerce")

def yes_no_to_binary(series):
    s = series.astype(str).str.strip().str.lower()
    out = pd.Series(np.nan, index=series.index, dtype="float")
    out[s == "yes"] = 1.0
    out[s == "no"] = 0.0
    return out

def file_exists_anywhere(filename):
    cands = [PROJECT_ROOT / filename, RAW_DIR / filename]
    for p in cands:
        if p.exists():
            return p
    return None

def copy_raw_files_into_raw_dir():
    for key, fname in FILE_MAP.items():
        src = file_exists_anywhere(fname)
        if src is None:
            raise FileNotFoundError(f"Required file not found: {fname}")
        dst = RAW_DIR / fname
        if src.resolve() != dst.resolve():
            if not dst.exists():
                shutil.copy2(src, dst)
                log(f"Copied {fname} to raw_data/")
        else:
            log(f"{fname} already in raw_data/")

copy_raw_files_into_raw_dir()
log("Logging initialized.")


In [ ]:

# ============================================================
# 3. Input readers and raw-table audit
# ============================================================
stage_name = "stage_01_raw_audit"
stage_obj = maybe_load_stage(stage_name)

if stage_obj is None:
    raw_tables = {
        "ASR": pd.read_csv(RAW_DIR / FILE_MAP["ASR"]),
        "FSR": pd.read_csv(RAW_DIR / FILE_MAP["FSR"]),
        "ION": pd.read_csv(RAW_DIR / FILE_MAP["ION"]),
        "CROSSWALK": pd.read_csv(RAW_DIR / FILE_MAP["CROSSWALK"]),
        "SCREENING": pd.read_csv(RAW_DIR / FILE_MAP["SCREENING"]),
    }

    raw_tables_str = {
        k: pd.read_csv(RAW_DIR / FILE_MAP[k], dtype=str)
        for k in ["ASR", "FSR", "ION"]
    }

    def audit_table(df_num, df_str, table_name):
        records = []
        for col in df_str.columns:
            s = df_str[col]
            s_nonnull = s.dropna().astype(str).str.strip()
            n = len(s_nonnull)
            numeric_count = pd.to_numeric(s_nonnull, errors="coerce").notna().sum() if n > 0 else 0
            unknown_count = s_nonnull.str.lower().isin(["unknown", "na", "nan", "none", ""]).sum() if n > 0 else 0
            mixed_type = (numeric_count > 0) and (numeric_count < n)
            records.append({
                "table": table_name,
                "column": col,
                "dtype_inferred": str(df_num[col].dtype),
                "n_rows": len(df_str),
                "nonnull_count": n,
                "numeric_like_count": int(numeric_count),
                "non_numeric_like_count": int(n - numeric_count),
                "unknown_like_count": int(unknown_count),
                "n_unique_nonnull": int(s_nonnull.nunique()),
                "mixed_type_flag": bool(mixed_type),
                "top_5_values": json.dumps(s_nonnull.value_counts().head(5).to_dict(), ensure_ascii=False),
            })
        return pd.DataFrame(records)

    audit_frames = []
    for name in ["ASR", "FSR", "ION"]:
        audit_frames.append(audit_table(raw_tables[name], raw_tables_str[name], name))

    audit_df = pd.concat(audit_frames, ignore_index=True)

    stage_obj = {
        "raw_tables": raw_tables,
        "raw_tables_str": raw_tables_str,
        "audit_df": audit_df,
    }
    save_stage(stage_name, stage_obj)
else:
    raw_tables = stage_obj["raw_tables"]
    raw_tables_str = stage_obj["raw_tables_str"]
    audit_df = stage_obj["audit_df"]

for name, df in raw_tables.items():
    log(f"{name}: shape={df.shape}")

export_df(audit_df, "raw_column_audit", index=False)
display(audit_df.head(20))


In [ ]:

# Quick audit summary
summary_rows = []
for name in ["ASR", "FSR", "ION", "CROSSWALK", "SCREENING"]:
    df = raw_tables[name]
    summary_rows.append({
        "table": name,
        "n_rows": df.shape[0],
        "n_cols": df.shape[1],
        "columns_preview": ", ".join(df.columns[:8]),
    })
summary_df = pd.DataFrame(summary_rows)
export_df(summary_df, "input_file_summary", index=False)
display(summary_df)


In [ ]:

# ============================================================
# 4. Normalization helpers
# ============================================================
def normalize_coreid(x):
    if pd.isna(x):
        return np.nan
    return str(x).strip()

def normalize_mofid_v1_base(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    return x.split(";")[0].strip()

def normalize_metal_types(x):
    if pd.isna(x):
        return np.nan
    parts = [p.strip() for p in str(x).split(",") if str(p).strip()]
    parts = sorted(parts)
    return ",".join(parts) if parts else np.nan

def split_metal_types(x):
    if pd.isna(x):
        return {
            "n_metals": np.nan,
            "is_mixed_metal": np.nan,
            "primary_metal": np.nan,
            "secondary_metal": np.nan,
            "metal_list": np.nan,
        }
    parts = [p.strip() for p in str(x).split(",") if str(p).strip()]
    parts_sorted = sorted(parts)
    return {
        "n_metals": len(parts_sorted),
        "is_mixed_metal": int(len(parts_sorted) > 1),
        "primary_metal": parts_sorted[0] if len(parts_sorted) >= 1 else np.nan,
        "secondary_metal": parts_sorted[1] if len(parts_sorted) >= 2 else np.nan,
        "metal_list": ",".join(parts_sorted) if parts_sorted else np.nan,
    }

def add_normalized_id_columns(df):
    df = df.copy()
    df["coreid_norm"] = df["coreid"].map(normalize_coreid) if "coreid" in df.columns else np.nan
    df["mofid1_base"] = df["mofid-v1"].map(normalize_mofid_v1_base) if "mofid-v1" in df.columns else np.nan
    df["metal_types_norm"] = df["Metal Types"].map(normalize_metal_types) if "Metal Types" in df.columns else np.nan
    metal_info = df["Metal Types"].map(split_metal_types).apply(pd.Series)
    df = pd.concat([df, metal_info], axis=1)
    return df

NUMERIC_COLUMNS_COMMON = [
    "LCD (Å)", "PLD (Å)", "LFPD (Å)", "Density (g/cm3)",
    "ASA (A2)", "ASA (m2/cm3)", "ASA (m2/g)",
    "NASA (A2)", "NASA (m2/cm3)", "NASA (m2/g)",
    "PV (A3)", "VF", "PV (cm3/g)", "NAV (A3)", "NAV_VF", "NPV (cm3/g)",
    "structure_dimension", "catenation", "dimension_by_topo", "hall", "number_spacegroup",
    "average_atomic_mass",
    "Heat_capacity@300K (J/g/K)", "std @ 300 K (J/g/K)",
    "Heat_capacity@350K (J/g/K)", "std @ 350 K (J/g/K)",
    "Heat_capacity@400K (J/g/K)", "std @ 400 K (J/g/K)",
    "k_cp (J/g/K/K)", "cp0 (J/g/K)", "natoms", "Year", "Time",
    "Thermal_stability (℃)", "Solvent_stability", "Water_stability"
]

CATEGORICAL_COLUMNS_COMMON = [
    "refcode", "name", "mofid-v1", "mofid-v2",
    "topology(SingleNodes)", "topology(SingleNodes)-zeo",
    "topology(AllNodes)", "topology(AllNodes)-zeo",
    "Metal Types", "Has OMS", "OMS Types",
    "Charge", "Source", "DOI", "Publication", "Extension",
    "unmodified", "KH_Classes", "memo"
]

def coerce_known_columns(df):
    df = df.copy()
    for col in NUMERIC_COLUMNS_COMMON:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    if "Has OMS" in df.columns:
        df["Has OMS_binary"] = yes_no_to_binary(df["Has OMS"])
    return df


In [ ]:

# ============================================================
# 5. Build clean dataset-specific tables
# ============================================================
stage_name = "stage_02_clean_tables"
stage_obj = maybe_load_stage(stage_name)

if stage_obj is None:
    clean_tables = {}

    for name in ["ASR", "FSR", "ION"]:
        df = raw_tables[name].copy()
        df = add_normalized_id_columns(df)
        df = coerce_known_columns(df)

        # flag membership in recommended screening list
        screening_ids = raw_tables["SCREENING"]["core-mof-id"].astype(str).str.strip().unique()
        screening_set = set(screening_ids)
        if "coreid" in df.columns:
            df["in_recommended_screening_list"] = df["coreid"].astype(str).str.strip().isin(screening_set).astype(int)
        else:
            df["in_recommended_screening_list"] = 0

        # Explicit task-ready variants for ASR and FSR will later remove unknown targets by task
        clean_tables[name] = df

    stage_obj = {"clean_tables": clean_tables}
    save_stage(stage_name, stage_obj)
else:
    clean_tables = stage_obj["clean_tables"]

asr_clean = clean_tables["ASR"].copy()
fsr_clean = clean_tables["FSR"].copy()
ion_clean = clean_tables["ION"].copy()

export_df(asr_clean, "asr_clean", index=False)
export_df(fsr_clean, "fsr_clean", index=False)
export_df(ion_clean, "ion_clean", index=False)

log(f"ASR clean shape: {asr_clean.shape}")
log(f"FSR clean shape: {fsr_clean.shape}")
log(f"ION clean shape: {ion_clean.shape}")

display(asr_clean.head())


In [ ]:

# Inspect target completeness
target_completeness = []
for name, df in [("ASR", asr_clean), ("FSR", fsr_clean), ("ION", ion_clean)]:
    for target in TARGETS:
        if target in df.columns:
            target_completeness.append({
                "dataset": name,
                "target": target,
                "n_rows": len(df),
                "n_numeric": int(df[target].notna().sum()),
                "n_missing_or_unknown": int(df[target].isna().sum()),
            })
target_completeness_df = pd.DataFrame(target_completeness)
export_df(target_completeness_df, "target_completeness", index=False)
display(target_completeness_df)


In [ ]:

# ============================================================
# 6. Descriptor regimes
# ============================================================
REGIMES = {
    "A_metal_only": [
        "primary_metal", "is_mixed_metal", "n_metals"
    ],
    "B_metal_oms": [
        "primary_metal", "is_mixed_metal", "n_metals",
        "Has OMS", "OMS Types"
    ],
    "C_metal_oms_context": [
        "primary_metal", "is_mixed_metal", "n_metals",
        "Has OMS", "OMS Types",
        "structure_dimension",
        "topology(SingleNodes)", "topology(AllNodes)",
        "catenation",
        "Density (g/cm3)", "LCD (Å)", "PLD (Å)", "VF", "PV (cm3/g)"
    ],
    "D_context_thermophysical": [
        "primary_metal", "is_mixed_metal", "n_metals",
        "Has OMS", "OMS Types",
        "structure_dimension",
        "topology(SingleNodes)", "topology(AllNodes)",
        "catenation",
        "Density (g/cm3)", "LCD (Å)", "PLD (Å)", "VF", "PV (cm3/g)",
        "average_atomic_mass",
        "Heat_capacity@300K (J/g/K)",
        "Heat_capacity@350K (J/g/K)",
        "Heat_capacity@400K (J/g/K)",
        "k_cp (J/g/K/K)", "cp0 (J/g/K)", "natoms"
    ],
}

def existing_features(df, feature_list):
    return [c for c in feature_list if c in df.columns]

regime_table = pd.DataFrame(
    [{"regime": k, "n_features_requested": len(v), "features": ", ".join(v)} for k, v in REGIMES.items()]
)
export_df(regime_table, "descriptor_regimes", index=False)
display(regime_table)


In [ ]:

# ============================================================
# 7. Task-ready subsets
# ============================================================
def make_task_table(df, dataset_name, target):
    work = df.copy()
    work = work[work[target].notna()].copy()
    work["dataset_name"] = dataset_name
    work["target_name"] = target
    return work

task_tables = {}
for dataset_name, df in [("ASR", asr_clean), ("FSR", fsr_clean)]:
    for target in TARGETS:
        task_tables[(dataset_name, target)] = make_task_table(df, dataset_name, target)

task_size_df = pd.DataFrame([
    {
        "dataset": ds,
        "target": target,
        "n_rows": len(task_tables[(ds, target)])
    }
    for ds, target in task_tables
])
export_df(task_size_df, "task_table_sizes", index=False)
display(task_size_df)


In [ ]:

# ============================================================
# 8. ASR–FSR matched pairs
# ============================================================
stage_name = "stage_03_matched_pairs"
stage_obj = maybe_load_stage(stage_name)

if stage_obj is None:
    crosswalk = raw_tables["CROSSWALK"].copy()
    crosswalk["ASR_id_norm"] = crosswalk["ASR_id"].astype(str).str.strip()
    crosswalk["FSR_id_norm"] = crosswalk["FSR_id"].astype(str).str.strip()

    asr_idx = asr_clean.copy()
    asr_idx["ASR_id_norm"] = asr_idx["coreid"].astype(str).str.strip()
    fsr_idx = fsr_clean.copy()
    fsr_idx["FSR_id_norm"] = fsr_idx["coreid"].astype(str).str.strip()

    matched = crosswalk.merge(
        asr_idx.add_prefix("ASR__"),
        left_on="ASR_id_norm",
        right_on="ASR__ASR_id_norm",
        how="left"
    ).merge(
        fsr_idx.add_prefix("FSR__"),
        left_on="FSR_id_norm",
        right_on="FSR__FSR_id_norm",
        how="left"
    )

    # keep rows where both sides successfully merge
    matched = matched[matched["ASR__coreid"].notna() & matched["FSR__coreid"].notna()].copy()

    stage_obj = {"matched_pairs": matched}
    save_stage(stage_name, stage_obj)
else:
    matched = stage_obj["matched_pairs"]

export_df(matched, "asr_fsr_matched_pairs", index=False)
log(f"Matched ASR–FSR pairs: {len(matched)}")
display(matched.head())


In [ ]:

# Pairwise consistency summary
pair_consistency_rows = []
for target in ["Solvent_stability", "Water_stability", "Thermal_stability (℃)"]:
    a = matched[f"ASR__{target}"]
    b = matched[f"FSR__{target}"]
    mask = a.notna() & b.notna()
    if mask.sum() >= 3:
        rho, p = spearmanr(a[mask], b[mask])
        pair_consistency_rows.append({
            "target": target,
            "n_pairs": int(mask.sum()),
            "spearman_rho": float(rho),
            "p_value": float(p),
            "mean_difference_ASR_minus_FSR": float((a[mask] - b[mask]).mean()),
            "std_difference_ASR_minus_FSR": float((a[mask] - b[mask]).std()),
        })

pair_consistency_df = pd.DataFrame(pair_consistency_rows)
export_df(pair_consistency_df, "pair_consistency_summary", index=False)
display(pair_consistency_df)


In [ ]:

# ============================================================
# 9. Preprocessing and model builders
# ============================================================
def split_feature_types(df, feature_cols):
    feature_cols = [c for c in feature_cols if c in df.columns]
    numeric_cols = []
    categorical_cols = []
    for c in feature_cols:
        if pd.api.types.is_numeric_dtype(df[c]):
            numeric_cols.append(c)
        else:
            categorical_cols.append(c)
    return numeric_cols, categorical_cols

def build_preprocessor(df, feature_cols):
    numeric_cols, categorical_cols = split_feature_types(df, feature_cols)

    numeric_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, numeric_cols),
            ("cat", categorical_pipe, categorical_cols),
        ],
        remainder="drop"
    )
    return preprocessor, numeric_cols, categorical_cols

def regression_models():
    return {
        "Ridge": Ridge(alpha=1.0, random_state=None),
        "RandomForest": RandomForestRegressor(
            n_estimators=400, random_state=RANDOM_STATE, n_jobs=-1, min_samples_leaf=2
        ),
        "HistGB": HistGradientBoostingRegressor(
            random_state=RANDOM_STATE, max_depth=6, learning_rate=0.05
        ),
    }

def classification_models():
    return {
        "Logistic": LogisticRegression(
            max_iter=2000, random_state=RANDOM_STATE
        ),
        "RandomForest": RandomForestClassifier(
            n_estimators=400, random_state=RANDOM_STATE, n_jobs=-1, min_samples_leaf=2
        ),
        "HistGB": HistGradientBoostingClassifier(
            random_state=RANDOM_STATE, max_depth=6, learning_rate=0.05
        ),
    }

def build_pipeline(df, feature_cols, estimator):
    preprocessor, numeric_cols, categorical_cols = build_preprocessor(df, feature_cols)
    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", estimator)
    ])
    return pipe, numeric_cols, categorical_cols

def regression_metrics(y_true, y_pred):
    rmse = mean_squared_error(y_true, y_pred, squared=False)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    rho = spearmanr(y_true, y_pred).statistic if len(y_true) >= 3 else np.nan
    return {"RMSE": rmse, "MAE": mae, "R2": r2, "Spearman": rho}

def classification_metrics(y_true, y_prob, cutoff=0.5):
    y_pred = (y_prob >= cutoff).astype(int)
    out = {
        "ROC_AUC": np.nan,
        "PR_AUC": np.nan,
        "Balanced_Accuracy": balanced_accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
    }
    if len(np.unique(y_true)) > 1:
        out["ROC_AUC"] = roc_auc_score(y_true, y_prob)
        out["PR_AUC"] = average_precision_score(y_true, y_prob)
    return out

def top_confidence_precision(y_true, y_prob, frac=TOP_CONFIDENCE_FRACTION):
    n = len(y_true)
    if n == 0:
        return np.nan
    k = max(1, int(math.ceil(frac * n)))
    idx = np.argsort(-y_prob)[:k]
    return y_true.iloc[idx].mean()


In [ ]:

# ============================================================
# 10. Split generators
# ============================================================
def get_random_splits(df, n_repeats=5, test_size=0.20, random_state=42):
    splits = []
    for i in range(n_repeats):
        train_idx, test_idx = train_test_split(
            np.arange(len(df)),
            test_size=test_size,
            random_state=random_state + i,
            shuffle=True
        )
        splits.append((f"random_repeat_{i+1}", train_idx, test_idx))
    return splits

def get_group_shuffle_splits(df, group_col, n_repeats=5, test_size=0.20, random_state=42):
    if group_col not in df.columns:
        return []
    groups = df[group_col].fillna("MISSING_GROUP").astype(str).values
    splitter = GroupShuffleSplit(n_splits=n_repeats, test_size=test_size, random_state=random_state)
    splits = []
    for i, (train_idx, test_idx) in enumerate(splitter.split(df, groups=groups), start=1):
        splits.append((f"group_{group_col}_{i}", train_idx, test_idx))
    return splits

def choose_grouped_split_candidates(df):
    candidates = []
    if "primary_metal" in df.columns:
        candidates.append("primary_metal")
    if "topology(AllNodes)" in df.columns:
        candidates.append("topology(AllNodes)")
    elif "topology(SingleNodes)" in df.columns:
        candidates.append("topology(SingleNodes)")
    if "Year" in df.columns:
        candidates.append("Year")
    return candidates


In [ ]:

# ============================================================
# 11. Regression benchmark runner
# ============================================================
stage_name = "stage_04_regression_results"
stage_obj = maybe_load_stage(stage_name)

if stage_obj is None:
    regression_rows = []
    regression_predictions = []

    for dataset_name in ["ASR", "FSR"]:
        for target in PRIMARY_REGRESSION_TARGETS + SUPPLEMENTARY_REGRESSION_TARGETS:
            df = task_tables[(dataset_name, target)].copy()
            if len(df) < 50:
                log(f"Skipping tiny task: {dataset_name} / {target}")
                continue

            all_splits = []
            all_splits.extend(get_random_splits(df, n_repeats=N_RANDOM_REPEATS, test_size=0.20, random_state=RANDOM_STATE))

            for group_col in choose_grouped_split_candidates(df):
                try:
                    all_splits.extend(get_group_shuffle_splits(df, group_col, n_repeats=3, test_size=0.20, random_state=RANDOM_STATE))
                except Exception as e:
                    log(f"Grouped split failed for {dataset_name}, {target}, {group_col}: {e}")

            for regime_name, regime_features in REGIMES.items():
                feature_cols = existing_features(df, regime_features)
                if len(feature_cols) == 0:
                    continue

                for model_name, model in regression_models().items():
                    for split_name, train_idx, test_idx in all_splits:
                        train_df = df.iloc[train_idx].copy()
                        test_df = df.iloc[test_idx].copy()

                        X_train = train_df[feature_cols]
                        y_train = train_df[target]
                        X_test = test_df[feature_cols]
                        y_test = test_df[target]

                        pipe, num_cols, cat_cols = build_pipeline(train_df, feature_cols, model)
                        pipe.fit(X_train, y_train)
                        y_pred = pipe.predict(X_test)

                        mets = regression_metrics(y_test, y_pred)

                        regression_rows.append({
                            "dataset": dataset_name,
                            "target": target,
                            "regime": regime_name,
                            "model": model_name,
                            "split_name": split_name,
                            "n_train": len(train_df),
                            "n_test": len(test_df),
                            "n_features_requested": len(regime_features),
                            "n_features_used": len(feature_cols),
                            "n_numeric_features": len(num_cols),
                            "n_categorical_features": len(cat_cols),
                            **mets
                        })

                        pred_df = pd.DataFrame({
                            "dataset": dataset_name,
                            "target": target,
                            "regime": regime_name,
                            "model": model_name,
                            "split_name": split_name,
                            "row_index": test_df.index,
                            "coreid": test_df["coreid"].values if "coreid" in test_df.columns else np.nan,
                            "y_true": y_test.values,
                            "y_pred": y_pred,
                            "primary_metal": test_df["primary_metal"].values if "primary_metal" in test_df.columns else np.nan,
                            "is_mixed_metal": test_df["is_mixed_metal"].values if "is_mixed_metal" in test_df.columns else np.nan,
                            "Has OMS": test_df["Has OMS"].values if "Has OMS" in test_df.columns else np.nan,
                            "topology(AllNodes)": test_df["topology(AllNodes)"].values if "topology(AllNodes)" in test_df.columns else np.nan,
                            "in_recommended_screening_list": test_df["in_recommended_screening_list"].values if "in_recommended_screening_list" in test_df.columns else 0,
                        })
                        regression_predictions.append(pred_df)

    regression_results_df = pd.DataFrame(regression_rows)
    regression_predictions_df = pd.concat(regression_predictions, ignore_index=True) if regression_predictions else pd.DataFrame()

    stage_obj = {
        "regression_results_df": regression_results_df,
        "regression_predictions_df": regression_predictions_df,
    }
    save_stage(stage_name, stage_obj)
else:
    regression_results_df = stage_obj["regression_results_df"]
    regression_predictions_df = stage_obj["regression_predictions_df"]

export_df(regression_results_df, "regression_results", index=False)
export_df(regression_predictions_df, "regression_predictions", index=False)

display(regression_results_df.head())


In [ ]:

# Regression summary table
regression_summary = (
    regression_results_df
    .groupby(["dataset", "target", "regime", "model"], as_index=False)
    .agg({
        "RMSE": "mean",
        "MAE": "mean",
        "R2": "mean",
        "Spearman": "mean",
    })
    .sort_values(["dataset", "target", "Spearman", "R2"], ascending=[True, True, False, False])
)
export_df(regression_summary, "regression_summary", index=False)
display(regression_summary.head(20))


In [ ]:

# ============================================================
# 12. Thresholded screening analysis
# ============================================================
stage_name = "stage_05_classification_results"
stage_obj = maybe_load_stage(stage_name)

if stage_obj is None:
    class_rows = []
    class_predictions = []

    for dataset_name in ["ASR", "FSR"]:
        for target in PRIMARY_REGRESSION_TARGETS:
            df = task_tables[(dataset_name, target)].copy()
            if len(df) < 50:
                continue

            all_splits = []
            all_splits.extend(get_random_splits(df, n_repeats=N_RANDOM_REPEATS, test_size=0.20, random_state=RANDOM_STATE))
            for group_col in choose_grouped_split_candidates(df):
                all_splits.extend(get_group_shuffle_splits(df, group_col, n_repeats=3, test_size=0.20, random_state=RANDOM_STATE))

            for cutoff in CLASSIFICATION_CUTOFFS:
                y_binary = (df[target] >= cutoff).astype(int)
                if y_binary.nunique() < 2:
                    log(f"Skipping cutoff {cutoff} for {dataset_name}/{target}: only one class present.")
                    continue
                df = df.copy()
                df["binary_target"] = y_binary

                for regime_name, regime_features in REGIMES.items():
                    feature_cols = existing_features(df, regime_features)
                    if len(feature_cols) == 0:
                        continue

                    for model_name, model in classification_models().items():
                        for split_name, train_idx, test_idx in all_splits:
                            train_df = df.iloc[train_idx].copy()
                            test_df = df.iloc[test_idx].copy()

                            # require both classes in train and test for robust scoring
                            if train_df["binary_target"].nunique() < 2 or test_df["binary_target"].nunique() < 2:
                                continue

                            X_train = train_df[feature_cols]
                            y_train = train_df["binary_target"]
                            X_test = test_df[feature_cols]
                            y_test = test_df["binary_target"]

                            pipe, num_cols, cat_cols = build_pipeline(train_df, feature_cols, model)
                            pipe.fit(X_train, y_train)

                            if hasattr(pipe, "predict_proba"):
                                y_prob = pipe.predict_proba(X_test)[:, 1]
                            else:
                                score = pipe.decision_function(X_test)
                                y_prob = 1.0 / (1.0 + np.exp(-score))

                            mets = classification_metrics(y_test, y_prob, cutoff=0.5)
                            tpp = top_confidence_precision(y_test.reset_index(drop=True), pd.Series(y_prob), frac=TOP_CONFIDENCE_FRACTION)

                            class_rows.append({
                                "dataset": dataset_name,
                                "target": target,
                                "screening_cutoff": cutoff,
                                "regime": regime_name,
                                "model": model_name,
                                "split_name": split_name,
                                "n_train": len(train_df),
                                "n_test": len(test_df),
                                "positive_rate_test": y_test.mean(),
                                "top_confidence_precision": tpp,
                                **mets
                            })

                            pred_df = pd.DataFrame({
                                "dataset": dataset_name,
                                "target": target,
                                "screening_cutoff": cutoff,
                                "regime": regime_name,
                                "model": model_name,
                                "split_name": split_name,
                                "row_index": test_df.index,
                                "coreid": test_df["coreid"].values,
                                "y_true_binary": y_test.values,
                                "y_prob": y_prob,
                                "primary_metal": test_df["primary_metal"].values if "primary_metal" in test_df.columns else np.nan,
                                "is_mixed_metal": test_df["is_mixed_metal"].values if "is_mixed_metal" in test_df.columns else np.nan,
                                "Has OMS": test_df["Has OMS"].values if "Has OMS" in test_df.columns else np.nan,
                                "topology(AllNodes)": test_df["topology(AllNodes)"].values if "topology(AllNodes)" in test_df.columns else np.nan,
                                "in_recommended_screening_list": test_df["in_recommended_screening_list"].values if "in_recommended_screening_list" in test_df.columns else 0,
                            })
                            class_predictions.append(pred_df)

    classification_results_df = pd.DataFrame(class_rows)
    classification_predictions_df = pd.concat(class_predictions, ignore_index=True) if class_predictions else pd.DataFrame()

    stage_obj = {
        "classification_results_df": classification_results_df,
        "classification_predictions_df": classification_predictions_df,
    }
    save_stage(stage_name, stage_obj)
else:
    classification_results_df = stage_obj["classification_results_df"]
    classification_predictions_df = stage_obj["classification_predictions_df"]

export_df(classification_results_df, "classification_results", index=False)
export_df(classification_predictions_df, "classification_predictions", index=False)

display(classification_results_df.head())


In [ ]:

classification_summary = (
    classification_results_df
    .groupby(["dataset", "target", "screening_cutoff", "regime", "model"], as_index=False)
    .agg({
        "ROC_AUC": "mean",
        "PR_AUC": "mean",
        "Balanced_Accuracy": "mean",
        "top_confidence_precision": "mean",
    })
    .sort_values(["dataset", "target", "screening_cutoff", "ROC_AUC"], ascending=[True, True, True, False])
)
export_df(classification_summary, "classification_summary", index=False)
display(classification_summary.head(20))


In [ ]:

# ============================================================
# 13. Transfer validation: ASR -> FSR and FSR -> ASR
# ============================================================
stage_name = "stage_06_transfer_validation"
stage_obj = maybe_load_stage(stage_name)

if stage_obj is None:
    transfer_rows = []

    for target in PRIMARY_REGRESSION_TARGETS + SUPPLEMENTARY_REGRESSION_TARGETS:
        train_asr = task_tables[("ASR", target)].copy()
        train_fsr = task_tables[("FSR", target)].copy()

        for regime_name, regime_features in REGIMES.items():
            # shared features only
            shared_features = [c for c in regime_features if c in train_asr.columns and c in train_fsr.columns]
            if len(shared_features) == 0:
                continue

            for model_name, model in regression_models().items():
                # ASR -> FSR
                pipe, _, _ = build_pipeline(train_asr, shared_features, model)
                pipe.fit(train_asr[shared_features], train_asr[target])
                pred_fsr = pipe.predict(train_fsr[shared_features])
                mets_f = regression_metrics(train_fsr[target], pred_fsr)
                transfer_rows.append({
                    "direction": "ASR_to_FSR",
                    "target": target,
                    "regime": regime_name,
                    "model": model_name,
                    "n_train": len(train_asr),
                    "n_test": len(train_fsr),
                    **mets_f
                })

                # FSR -> ASR
                pipe, _, _ = build_pipeline(train_fsr, shared_features, model)
                pipe.fit(train_fsr[shared_features], train_fsr[target])
                pred_asr = pipe.predict(train_asr[shared_features])
                mets_a = regression_metrics(train_asr[target], pred_asr)
                transfer_rows.append({
                    "direction": "FSR_to_ASR",
                    "target": target,
                    "regime": regime_name,
                    "model": model_name,
                    "n_train": len(train_fsr),
                    "n_test": len(train_asr),
                    **mets_a
                })

    transfer_results_df = pd.DataFrame(transfer_rows)
    stage_obj = {"transfer_results_df": transfer_results_df}
    save_stage(stage_name, stage_obj)
else:
    transfer_results_df = stage_obj["transfer_results_df"]

export_df(transfer_results_df, "transfer_results", index=False)
display(transfer_results_df.head(20))


In [ ]:

# ============================================================
# 14. Subgroup performance and failure analysis
# ============================================================
def subgroup_metric_table(pred_df, group_col):
    rows = []
    if pred_df.empty or group_col not in pred_df.columns:
        return pd.DataFrame()
    for key, sub in pred_df.groupby(group_col):
        if len(sub) < 8:
            continue
        rows.append({
            "group_col": group_col,
            "group_value": key,
            "n": len(sub),
            "MAE": mean_absolute_error(sub["y_true"], sub["y_pred"]),
            "RMSE": mean_squared_error(sub["y_true"], sub["y_pred"], squared=False),
            "Spearman": spearmanr(sub["y_true"], sub["y_pred"]).statistic if len(sub) >= 3 else np.nan
        })
    return pd.DataFrame(rows)

# pick a "best" regression setting for chemistry-facing analysis
if not regression_summary.empty:
    best_regression_rows = (
        regression_summary.sort_values(["dataset", "target", "Spearman", "R2"], ascending=[True, True, False, False])
        .groupby(["dataset", "target"], as_index=False)
        .head(1)
    )
else:
    best_regression_rows = pd.DataFrame()

chemistry_subgroup_frames = []
failure_frames = []

for _, row in best_regression_rows.iterrows():
    dataset_name = row["dataset"]
    target = row["target"]
    regime = row["regime"]
    model = row["model"]

    pred = regression_predictions_df[
        (regression_predictions_df["dataset"] == dataset_name) &
        (regression_predictions_df["target"] == target) &
        (regression_predictions_df["regime"] == regime) &
        (regression_predictions_df["model"] == model)
    ].copy()

    pred["abs_error"] = (pred["y_true"] - pred["y_pred"]).abs()

    for group_col in ["primary_metal", "is_mixed_metal", "Has OMS", "topology(AllNodes)", "in_recommended_screening_list"]:
        gdf = subgroup_metric_table(pred, group_col)
        if not gdf.empty:
            gdf["dataset"] = dataset_name
            gdf["target"] = target
            gdf["regime"] = regime
            gdf["model"] = model
            chemistry_subgroup_frames.append(gdf)

    # worst-predicted cases
    fail = pred.sort_values("abs_error", ascending=False).head(50).copy()
    fail["dataset"] = dataset_name
    fail["target"] = target
    fail["regime"] = regime
    fail["model"] = model
    failure_frames.append(fail)

chemistry_subgroup_df = pd.concat(chemistry_subgroup_frames, ignore_index=True) if chemistry_subgroup_frames else pd.DataFrame()
failure_cases_df = pd.concat(failure_frames, ignore_index=True) if failure_frames else pd.DataFrame()

export_df(best_regression_rows, "best_regression_rows", index=False)
export_df(chemistry_subgroup_df, "chemistry_subgroup_performance", index=False)
export_df(failure_cases_df, "failure_cases_top50", index=False)

display(best_regression_rows)
display(chemistry_subgroup_df.head(20))


In [ ]:

# ============================================================
# 15. Permutation importance (interpretable, lightweight)
# ============================================================
stage_name = "stage_07_permutation_importance"
stage_obj = maybe_load_stage(stage_name)

if stage_obj is None:
    importance_rows = []

    for _, row in best_regression_rows.iterrows():
        dataset_name = row["dataset"]
        target = row["target"]
        regime = row["regime"]
        model_name = row["model"]

        df = task_tables[(dataset_name, target)].copy()
        feature_cols = existing_features(df, REGIMES[regime])

        # use a simple held-out random split for importance
        train_idx, test_idx = train_test_split(
            np.arange(len(df)), test_size=0.20, random_state=RANDOM_STATE, shuffle=True
        )
        train_df = df.iloc[train_idx].copy()
        test_df = df.iloc[test_idx].copy()

        model = regression_models()[model_name]
        pipe, _, _ = build_pipeline(train_df, feature_cols, model)
        pipe.fit(train_df[feature_cols], train_df[target])

        result = permutation_importance(
            pipe, test_df[feature_cols], test_df[target],
            n_repeats=12, random_state=RANDOM_STATE, scoring="neg_mean_absolute_error"
        )

        for feat, mean_imp, std_imp in zip(feature_cols, result.importances_mean, result.importances_std):
            importance_rows.append({
                "dataset": dataset_name,
                "target": target,
                "regime": regime,
                "model": model_name,
                "feature": feat,
                "importance_mean": mean_imp,
                "importance_std": std_imp,
            })

    permutation_importance_df = pd.DataFrame(importance_rows)
    stage_obj = {"permutation_importance_df": permutation_importance_df}
    save_stage(stage_name, stage_obj)
else:
    permutation_importance_df = stage_obj["permutation_importance_df"]

export_df(permutation_importance_df, "permutation_importance", index=False)
display(permutation_importance_df.sort_values("importance_mean", ascending=False).head(20))


In [ ]:

# ============================================================
# 16. Calibration, coverage, shortlist analysis
# ============================================================
stage_name = "stage_08_calibration_and_screening"
stage_obj = maybe_load_stage(stage_name)

if stage_obj is None:
    calibration_rows = []
    coverage_rows = []
    shortlist_rows = []

    if not classification_summary.empty:
        best_classification_rows = (
            classification_summary
            .sort_values(["dataset", "target", "screening_cutoff", "ROC_AUC"], ascending=[True, True, True, False])
            .groupby(["dataset", "target", "screening_cutoff"], as_index=False)
            .head(1)
        )
    else:
        best_classification_rows = pd.DataFrame()

    for _, row in best_classification_rows.iterrows():
        dataset_name = row["dataset"]
        target = row["target"]
        cutoff = row["screening_cutoff"]
        regime = row["regime"]
        model = row["model"]

        pred = classification_predictions_df[
            (classification_predictions_df["dataset"] == dataset_name) &
            (classification_predictions_df["target"] == target) &
            (classification_predictions_df["screening_cutoff"] == cutoff) &
            (classification_predictions_df["regime"] == regime) &
            (classification_predictions_df["model"] == model)
        ].copy()

        if pred.empty:
            continue

        # calibration bins
        try:
            frac_pos, mean_pred = calibration_curve(pred["y_true_binary"], pred["y_prob"], n_bins=10, strategy="quantile")
            for mp, fp in zip(mean_pred, frac_pos):
                calibration_rows.append({
                    "dataset": dataset_name,
                    "target": target,
                    "screening_cutoff": cutoff,
                    "regime": regime,
                    "model": model,
                    "mean_predicted_probability": mp,
                    "fraction_positive": fp,
                })
        except Exception as e:
            log(f"Calibration failed for {dataset_name}/{target}/{cutoff}: {e}")

        # coverage-retention
        pred = pred.sort_values("y_prob", ascending=False).reset_index(drop=True)
        n = len(pred)
        for cov in np.linspace(0.1, 1.0, 10):
            k = max(1, int(round(cov * n)))
            retained = pred.head(k)
            coverage_rows.append({
                "dataset": dataset_name,
                "target": target,
                "screening_cutoff": cutoff,
                "regime": regime,
                "model": model,
                "coverage": cov,
                "retained_n": k,
                "observed_positive_rate": retained["y_true_binary"].mean(),
                "precision_at_retained": retained["y_true_binary"].mean(),
            })

        # shortlist / recommended subset performance
        for subset_name, subset_df in {
            "full": pred,
            "recommended_screening_only": pred[pred["in_recommended_screening_list"] == 1].copy()
        }.items():
            if len(subset_df) < 10 or subset_df["y_true_binary"].nunique() < 2:
                continue
            shortlist_rows.append({
                "dataset": dataset_name,
                "target": target,
                "screening_cutoff": cutoff,
                "subset": subset_name,
                "n": len(subset_df),
                "ROC_AUC": roc_auc_score(subset_df["y_true_binary"], subset_df["y_prob"]),
                "PR_AUC": average_precision_score(subset_df["y_true_binary"], subset_df["y_prob"]),
                "top_decile_positive_rate": subset_df.sort_values("y_prob", ascending=False).head(max(1, int(0.1 * len(subset_df))))["y_true_binary"].mean(),
                "overall_positive_rate": subset_df["y_true_binary"].mean(),
            })

    calibration_df = pd.DataFrame(calibration_rows)
    coverage_df = pd.DataFrame(coverage_rows)
    shortlist_df = pd.DataFrame(shortlist_rows)

    stage_obj = {
        "best_classification_rows": best_classification_rows,
        "calibration_df": calibration_df,
        "coverage_df": coverage_df,
        "shortlist_df": shortlist_df,
    }
    save_stage(stage_name, stage_obj)
else:
    best_classification_rows = stage_obj["best_classification_rows"]
    calibration_df = stage_obj["calibration_df"]
    coverage_df = stage_obj["coverage_df"]
    shortlist_df = stage_obj["shortlist_df"]

export_df(best_classification_rows, "best_classification_rows", index=False)
export_df(calibration_df, "calibration_curve_data", index=False)
export_df(coverage_df, "coverage_curve_data", index=False)
export_df(shortlist_df, "shortlist_subset_performance", index=False)

display(best_classification_rows)
display(shortlist_df.head(20))



## Figure generation

The next cells generate the main figures and the SI-style figures.  
All figures are saved to `project15_outputs/figures/` as PNG files.


In [ ]:

# ============================================================
# 17. Figure helpers
# ============================================================
plt.rcParams["figure.dpi"] = 140
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["font.size"] = 10

def savefig(path_stem):
    png_path = FIG_DIR / f"{path_stem}.png"
    plt.tight_layout()
    plt.savefig(png_path, bbox_inches="tight")
    log(f"Saved figure: {png_path.name}")
    plt.show()

def count_overlap():
    asr_ids = set(asr_clean["coreid_norm"].dropna())
    fsr_ids = set(fsr_clean["coreid_norm"].dropna())
    both = asr_ids & fsr_ids
    return {
        "ASR_only": len(asr_ids - fsr_ids),
        "FSR_only": len(fsr_ids - asr_ids),
        "Matched_by_coreid": len(both),
        "Matched_by_crosswalk": len(matched),
    }

def top_n_counts(series, n=12):
    vc = series.astype(str).fillna("MISSING").value_counts().head(n)
    return vc.index.tolist(), vc.values.tolist()


In [ ]:

# ============================================================
# Figure 1. Dataset architecture and benchmark logic
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# (a) dataset flow diagram
ax = axes[0, 0]
ax.axis("off")
flow_text = (
    "ASR_data_SI_20250204.csv  ─┐\n"
    "FSR_data_SI_20250204.csv  ─┼─> clean tables ─> descriptor regimes A–D ─> regression / screening / subgroup analysis\n"
    "ION_data_SI_20250204.csv  ─┘            │\n"
    "ASR_FSR_check.csv ----------------------┤─> matched ASR–FSR trustworthiness diagnostics\n"
    "12089-recommended-screening-list.csv ---┘─> shortlist / applicability analysis"
)
ax.text(0.02, 0.98, flow_text, va="top", ha="left", family="monospace", fontsize=10)
ax.set_title("(a) Dataset flow diagram", loc="left", fontweight="bold")

# (b) target distributions
ax = axes[0, 1]
for dataset_name, df in [("ASR", asr_clean), ("FSR", fsr_clean)]:
    for target, ls in zip(TARGETS, ["-", "--", ":"]):
        vals = df[target].dropna()
        ax.hist(vals, bins=30, histtype="step", label=f"{dataset_name}: {target}", linewidth=1.5)
ax.set_xlabel("Target value")
ax.set_ylabel("Count")
ax.set_title("(b) Target distributions", loc="left", fontweight="bold")
ax.legend(fontsize=7)

# (c) overlap map
ax = axes[1, 0]
ov = count_overlap()
labels = list(ov.keys())
vals = list(ov.values())
ax.bar(labels, vals)
ax.set_ylabel("Count")
ax.set_title("(c) Overlap map", loc="left", fontweight="bold")
ax.tick_params(axis="x", rotation=20)

# (d) descriptor ladder
ax = axes[1, 1]
ax.axis("off")
ladder_text = (
    "Regime A: metal identity only\n"
    "  primary_metal, is_mixed_metal, n_metals\n\n"
    "Regime B: A + OMS\n"
    "  Has OMS, OMS Types\n\n"
    "Regime C: B + minimal structural context\n"
    "  dimension, topology, catenation, density, LCD, PLD, VF, PV\n\n"
    "Regime D: C + thermophysical terms\n"
    "  average_atomic_mass, heat capacities, natoms"
)
ax.text(0.02, 0.98, ladder_text, va="top", ha="left", fontsize=10)
ax.set_title("(d) Descriptor ladder", loc="left", fontweight="bold")

savefig("Figure_1_dataset_architecture")


In [ ]:

# ============================================================
# Figure 2. Are ASR and FSR mutually consistent?
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

# (a) paired scatter solvent
ax = axes[0, 0]
x = matched["ASR__Solvent_stability"]
y = matched["FSR__Solvent_stability"]
mask = x.notna() & y.notna()
ax.scatter(x[mask], y[mask], s=12, alpha=0.7)
lims = [min(x[mask].min(), y[mask].min()), max(x[mask].max(), y[mask].max())]
ax.plot(lims, lims, linewidth=1)
rho = spearmanr(x[mask], y[mask]).statistic if mask.sum() >= 3 else np.nan
ax.set_xlabel("ASR solvent stability")
ax.set_ylabel("FSR solvent stability")
ax.set_title(f"(a) Solvent stability pair scatter (ρ={rho:.2f})", loc="left", fontweight="bold")

# (b) paired scatter water
ax = axes[0, 1]
x = matched["ASR__Water_stability"]
y = matched["FSR__Water_stability"]
mask = x.notna() & y.notna()
ax.scatter(x[mask], y[mask], s=12, alpha=0.7)
lims = [min(x[mask].min(), y[mask].min()), max(x[mask].max(), y[mask].max())]
ax.plot(lims, lims, linewidth=1)
rho = spearmanr(x[mask], y[mask]).statistic if mask.sum() >= 3 else np.nan
ax.set_xlabel("ASR water stability")
ax.set_ylabel("FSR water stability")
ax.set_title(f"(b) Water stability pair scatter (ρ={rho:.2f})", loc="left", fontweight="bold")

# (c) threshold agreement heatmap
ax = axes[1, 0]
heat = []
for c1 in CLASSIFICATION_CUTOFFS:
    row = []
    for target in ["Solvent_stability", "Water_stability"]:
        a = (matched[f"ASR__{target}"] >= c1).astype(float)
        b = (matched[f"FSR__{target}"] >= c1).astype(float)
        mask = matched[f"ASR__{target}"].notna() & matched[f"FSR__{target}"].notna()
        agree = (a[mask] == b[mask]).mean()
        row.append(agree)
    heat.append(row)
heat = np.array(heat)
im = ax.imshow(heat, aspect="auto")
ax.set_xticks([0, 1], labels=["Solvent", "Water"])
ax.set_yticks(range(len(CLASSIFICATION_CUTOFFS)), labels=[str(c) for c in CLASSIFICATION_CUTOFFS])
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        ax.text(j, i, f"{heat[i,j]:.2f}", ha="center", va="center")
ax.set_title("(c) Threshold agreement heatmap", loc="left", fontweight="bold")
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

# (d) Bland–Altman / difference plot
ax = axes[1, 1]
target = "Water_stability"
a = matched[f"ASR__{target}"]
b = matched[f"FSR__{target}"]
mask = a.notna() & b.notna()
mean_vals = (a[mask] + b[mask]) / 2
diff_vals = a[mask] - b[mask]
ax.scatter(mean_vals, diff_vals, s=12, alpha=0.7)
m = diff_vals.mean()
s = diff_vals.std()
ax.axhline(m, linestyle="-", linewidth=1)
ax.axhline(m + 1.96*s, linestyle="--", linewidth=1)
ax.axhline(m - 1.96*s, linestyle="--", linewidth=1)
ax.set_xlabel("Mean(ASR, FSR)")
ax.set_ylabel("ASR - FSR")
ax.set_title("(d) Bland–Altman style plot (water stability)", loc="left", fontweight="bold")

savefig("Figure_2_asr_fsr_consistency")


In [ ]:

# ============================================================
# Figure 3. Descriptor sufficiency across tasks
# ============================================================
fig, axes = plt.subplots(3, 2, figsize=(14, 14))

# (a) solvent regression
ax = axes[0, 0]
tmp = regression_summary[(regression_summary["dataset"] == "ASR") & (regression_summary["target"] == "Solvent_stability")]
for model_name in tmp["model"].unique():
    sub = tmp[tmp["model"] == model_name]
    ax.plot(sub["regime"], sub["Spearman"], marker="o", label=model_name)
ax.set_title("(a) Solvent stability regression (ASR)", loc="left", fontweight="bold")
ax.set_ylabel("Mean Spearman")
ax.tick_params(axis="x", rotation=25)
ax.legend(fontsize=8)

# (b) water regression
ax = axes[0, 1]
tmp = regression_summary[(regression_summary["dataset"] == "ASR") & (regression_summary["target"] == "Water_stability")]
for model_name in tmp["model"].unique():
    sub = tmp[tmp["model"] == model_name]
    ax.plot(sub["regime"], sub["Spearman"], marker="o", label=model_name)
ax.set_title("(b) Water stability regression (ASR)", loc="left", fontweight="bold")
ax.set_ylabel("Mean Spearman")
ax.tick_params(axis="x", rotation=25)
ax.legend(fontsize=8)

# (c) screening ROC-AUC
ax = axes[1, 0]
tmp = classification_summary[(classification_summary["dataset"] == "ASR") & (classification_summary["target"] == "Water_stability")]
for regime in tmp["regime"].unique():
    sub = tmp[tmp["regime"] == regime].groupby("screening_cutoff", as_index=False)["ROC_AUC"].mean()
    ax.plot(sub["screening_cutoff"], sub["ROC_AUC"], marker="o", label=regime)
ax.set_title("(c) ROC-AUC across cutoffs", loc="left", fontweight="bold")
ax.set_xlabel("Cutoff")
ax.set_ylabel("ROC-AUC")
ax.legend(fontsize=7)

# (d) PR-AUC
ax = axes[1, 1]
tmp = classification_summary[(classification_summary["dataset"] == "ASR") & (classification_summary["target"] == "Water_stability")]
for regime in tmp["regime"].unique():
    sub = tmp[tmp["regime"] == regime].groupby("screening_cutoff", as_index=False)["PR_AUC"].mean()
    ax.plot(sub["screening_cutoff"], sub["PR_AUC"], marker="o", label=regime)
ax.set_title("(d) PR-AUC across cutoffs", loc="left", fontweight="bold")
ax.set_xlabel("Cutoff")
ax.set_ylabel("PR-AUC")
ax.legend(fontsize=7)

# (e) grouped-split performance drop
ax = axes[2, 0]
tmp = regression_results_df[(regression_results_df["dataset"] == "ASR") & (regression_results_df["target"] == "Water_stability")]
grp = tmp.assign(split_type=np.where(tmp["split_name"].str.startswith("random"), "random", "grouped"))
plot_df = grp.groupby(["regime", "split_type"], as_index=False)["Spearman"].mean()
for split_type in plot_df["split_type"].unique():
    sub = plot_df[plot_df["split_type"] == split_type]
    ax.plot(sub["regime"], sub["Spearman"], marker="o", label=split_type)
ax.set_title("(e) Grouped-split performance drop", loc="left", fontweight="bold")
ax.set_ylabel("Mean Spearman")
ax.tick_params(axis="x", rotation=25)
ax.legend()

# (f) transfer result
ax = axes[2, 1]
tmp = transfer_results_df[transfer_results_df["target"] == "Water_stability"]
for direction in tmp["direction"].unique():
    sub = tmp[tmp["direction"] == direction].groupby("regime", as_index=False)["Spearman"].mean()
    ax.plot(sub["regime"], sub["Spearman"], marker="o", label=direction)
ax.set_title("(f) Transfer validation", loc="left", fontweight="bold")
ax.set_ylabel("Spearman")
ax.tick_params(axis="x", rotation=25)
ax.legend(fontsize=8)

savefig("Figure_3_descriptor_sufficiency")


In [ ]:

# ============================================================
# Figure 4. Chemistry regime map for MOF chemists
# ============================================================
fig, axes = plt.subplots(3, 2, figsize=(14, 14))

# choose best ASR water prediction as chemistry-facing example
if not best_regression_rows.empty:
    row = best_regression_rows[
        (best_regression_rows["dataset"] == "ASR") &
        (best_regression_rows["target"] == "Water_stability")
    ].head(1)
    if len(row) == 1:
        row = row.iloc[0]
        pred = regression_predictions_df[
            (regression_predictions_df["dataset"] == row["dataset"]) &
            (regression_predictions_df["target"] == row["target"]) &
            (regression_predictions_df["regime"] == row["regime"]) &
            (regression_predictions_df["model"] == row["model"])
        ].copy()
        pred["abs_error"] = (pred["y_true"] - pred["y_pred"]).abs()

        # (a) performance by primary metal
        ax = axes[0, 0]
        g = pred.groupby("primary_metal").agg(n=("abs_error", "size"), MAE=("abs_error", "mean")).reset_index()
        g = g[g["n"] >= 8].sort_values("MAE").head(15)
        ax.barh(g["primary_metal"], g["MAE"])
        ax.set_xlabel("MAE")
        ax.set_title("(a) Performance by primary metal", loc="left", fontweight="bold")

        # (b) mixed-metal vs single-metal
        ax = axes[0, 1]
        g = pred.groupby("is_mixed_metal").agg(MAE=("abs_error", "mean"), n=("abs_error", "size")).reset_index()
        ax.bar(g["is_mixed_metal"].astype(str), g["MAE"])
        ax.set_xlabel("is_mixed_metal")
        ax.set_ylabel("MAE")
        ax.set_title("(b) Mixed-metal vs single-metal", loc="left", fontweight="bold")

        # (c) OMS vs non-OMS
        ax = axes[1, 0]
        g = pred.groupby("Has OMS").agg(MAE=("abs_error", "mean"), n=("abs_error", "size")).reset_index()
        ax.bar(g["Has OMS"].astype(str), g["MAE"])
        ax.set_xlabel("Has OMS")
        ax.set_ylabel("MAE")
        ax.set_title("(c) OMS vs non-OMS", loc="left", fontweight="bold")

        # (d) median stability by metal × OMS
        ax = axes[1, 1]
        base = asr_clean[["primary_metal", "Has OMS", "Water_stability"]].dropna()
        top_metals = base["primary_metal"].value_counts().head(10).index
        pivot = (
            base[base["primary_metal"].isin(top_metals)]
            .pivot_table(index="primary_metal", columns="Has OMS", values="Water_stability", aggfunc="median")
        )
        im = ax.imshow(pivot.values, aspect="auto")
        ax.set_yticks(range(len(pivot.index)), labels=pivot.index)
        ax.set_xticks(range(len(pivot.columns)), labels=pivot.columns)
        for i in range(pivot.shape[0]):
            for j in range(pivot.shape[1]):
                val = pivot.iloc[i, j]
                ax.text(j, i, "" if pd.isna(val) else f"{val:.2f}", ha="center", va="center", fontsize=8)
        ax.set_title("(d) Median stability by metal × OMS", loc="left", fontweight="bold")
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

        # (e) topology-stratified performance
        ax = axes[2, 0]
        g = pred.groupby("topology(AllNodes)").agg(n=("abs_error", "size"), MAE=("abs_error", "mean")).reset_index()
        g = g[g["n"] >= 8].sort_values("MAE").head(15)
        ax.barh(g["topology(AllNodes)"], g["MAE"])
        ax.set_xlabel("MAE")
        ax.set_title("(e) Topology-stratified performance", loc="left", fontweight="bold")

        # (f) failure-enrichment map
        ax = axes[2, 1]
        fail = pred.sort_values("abs_error", ascending=False).head(100)
        fail_counts = fail["primary_metal"].fillna("MISSING").value_counts().head(12)
        ax.barh(fail_counts.index, fail_counts.values)
        ax.set_xlabel("Count among worst-predicted cases")
        ax.set_title("(f) Failure-enrichment map", loc="left", fontweight="bold")
    else:
        for ax in axes.ravel():
            ax.axis("off")
            ax.text(0.5, 0.5, "No best regression row available.", ha="center", va="center")
else:
    for ax in axes.ravel():
        ax.axis("off")
        ax.text(0.5, 0.5, "No best regression row available.", ha="center", va="center")

savefig("Figure_4_chemistry_regime_map")


In [ ]:

# ============================================================
# Figure 5. Practical screening and confidence
# ============================================================
fig, axes = plt.subplots(3, 2, figsize=(14, 14))

row = best_classification_rows[
    (best_classification_rows["dataset"] == "ASR") &
    (best_classification_rows["target"] == "Water_stability") &
    (best_classification_rows["screening_cutoff"] == 0.7)
].head(1)

if len(row) == 1:
    row = row.iloc[0]
    cal = calibration_df[
        (calibration_df["dataset"] == row["dataset"]) &
        (calibration_df["target"] == row["target"]) &
        (calibration_df["screening_cutoff"] == row["screening_cutoff"]) &
        (calibration_df["regime"] == row["regime"]) &
        (calibration_df["model"] == row["model"])
    ]
    cov = coverage_df[
        (coverage_df["dataset"] == row["dataset"]) &
        (coverage_df["target"] == row["target"]) &
        (coverage_df["screening_cutoff"] == row["screening_cutoff"]) &
        (coverage_df["regime"] == row["regime"]) &
        (coverage_df["model"] == row["model"])
    ]
    pred = classification_predictions_df[
        (classification_predictions_df["dataset"] == row["dataset"]) &
        (classification_predictions_df["target"] == row["target"]) &
        (classification_predictions_df["screening_cutoff"] == row["screening_cutoff"]) &
        (classification_predictions_df["regime"] == row["regime"]) &
        (classification_predictions_df["model"] == row["model"])
    ].copy()

    # (a) calibration
    ax = axes[0, 0]
    ax.plot([0,1], [0,1], linestyle="--", linewidth=1)
    ax.plot(cal["mean_predicted_probability"], cal["fraction_positive"], marker="o")
    ax.set_xlabel("Predicted probability")
    ax.set_ylabel("Observed positive fraction")
    ax.set_title("(a) Calibration curve", loc="left", fontweight="bold")

    # (b) confidence vs retained coverage
    ax = axes[0, 1]
    ax.plot(cov["coverage"], cov["precision_at_retained"], marker="o")
    ax.set_xlabel("Retained coverage")
    ax.set_ylabel("Observed positive fraction")
    ax.set_title("(b) Confidence vs retained coverage", loc="left", fontweight="bold")

    # (c) enrichment in top predicted decile
    ax = axes[1, 0]
    pred = pred.sort_values("y_prob", ascending=False).reset_index(drop=True)
    n = len(pred)
    top_decile = pred.head(max(1, int(0.1*n)))
    vals = [pred["y_true_binary"].mean(), top_decile["y_true_binary"].mean()]
    ax.bar(["Overall", "Top decile"], vals)
    ax.set_ylabel("Positive fraction")
    ax.set_title("(c) Enrichment in top predicted decile", loc="left", fontweight="bold")

    # (d) recommended-screening subset performance
    ax = axes[1, 1]
    tmp = shortlist_df[
        (shortlist_df["dataset"] == row["dataset"]) &
        (shortlist_df["target"] == row["target"]) &
        (shortlist_df["screening_cutoff"] == row["screening_cutoff"])
    ]
    if not tmp.empty:
        ax.bar(tmp["subset"], tmp["ROC_AUC"])
        ax.set_ylabel("ROC-AUC")
        ax.set_title("(d) Recommended-screening subset", loc="left", fontweight="bold")
        ax.tick_params(axis="x", rotation=20)

    # (e) example shortlist workflow
    ax = axes[2, 0]
    ax.axis("off")
    text = (
        "Example shortlist workflow\n\n"
        "1. Predict stability probability for all MOFs\n"
        "2. Restrict to recommended-screening subset\n"
        "3. Rank by predicted probability\n"
        "4. Keep top-confidence decile or top-N entries\n"
        "5. Review chemistry flags:\n"
        "   • metal family\n"
        "   • OMS\n"
        "   • mixed-metal status\n"
        "   • topology\n"
        "6. Experimental or simulation follow-up"
    )
    ax.text(0.02, 0.98, text, va="top", ha="left")
    ax.set_title("(e) Example shortlist workflow", loc="left", fontweight="bold")

    # (f) predicted probability distribution
    ax = axes[2, 1]
    ax.hist(pred["y_prob"], bins=25)
    ax.set_xlabel("Predicted probability")
    ax.set_ylabel("Count")
    ax.set_title("(f) Predicted probability distribution", loc="left", fontweight="bold")
else:
    for ax in axes.ravel():
        ax.axis("off")
        ax.text(0.5, 0.5, "No classification result available for ASR water stability cutoff 0.7", ha="center", va="center")

savefig("Figure_5_practical_screening")


In [ ]:

# ============================================================
# SI Figure set
# ============================================================

# S1 full missingness matrix and data audit (heatmap-style by column)
for dataset_name, df in [("ASR", asr_clean), ("FSR", fsr_clean), ("ION", ion_clean)]:
    fig, ax = plt.subplots(figsize=(16, 4))
    miss = df.isna().mean().sort_values(ascending=False)
    ax.bar(range(len(miss)), miss.values)
    ax.set_xticks(range(len(miss)))
    ax.set_xticklabels(miss.index, rotation=90, fontsize=7)
    ax.set_ylabel("Missing fraction")
    ax.set_title(f"S1/SI data missingness — {dataset_name}", loc="left", fontweight="bold")
    savefig(f"SI_{dataset_name}_missingness")

# S2 distributions of major numeric descriptors
major_numeric = ["Density (g/cm3)", "LCD (Å)", "PLD (Å)", "VF", "PV (cm3/g)", "average_atomic_mass", "natoms"]
fig, axes = plt.subplots(len(major_numeric), 1, figsize=(8, 2.6*len(major_numeric)))
for ax, col in zip(axes, major_numeric):
    vals = asr_clean[col].dropna()
    ax.hist(vals, bins=30)
    ax.set_title(f"S2 distribution — {col}", loc="left", fontweight="bold")
savefig("SI_descriptor_distributions_ASR")

# S3 correlation matrix among pore/context descriptors
corr_cols = ["Density (g/cm3)", "LCD (Å)", "PLD (Å)", "VF", "PV (cm3/g)", "average_atomic_mass", "natoms"]
corr = asr_clean[corr_cols].corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr.values, aspect="auto")
ax.set_xticks(range(len(corr.columns)), labels=corr.columns, rotation=90)
ax.set_yticks(range(len(corr.index)), labels=corr.index)
for i in range(corr.shape[0]):
    for j in range(corr.shape[1]):
        ax.text(j, i, f"{corr.iloc[i,j]:.2f}", ha="center", va="center", fontsize=8)
ax.set_title("S3 Correlation matrix among pore/context descriptors", loc="left", fontweight="bold")
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
savefig("SI_correlation_matrix_ASR")

# S4 metal frequency and mixed-metal breakdown
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
metal_counts = asr_clean["primary_metal"].value_counts().head(15)
axes[0].barh(metal_counts.index, metal_counts.values)
axes[0].set_title("S4 Primary metal frequency (ASR)", loc="left", fontweight="bold")
mix = asr_clean["is_mixed_metal"].value_counts(dropna=False)
axes[1].bar(mix.index.astype(str), mix.values)
axes[1].set_title("S4 Mixed-metal breakdown (ASR)", loc="left", fontweight="bold")
savefig("SI_metal_frequency_breakdown")

# S5 threshold robustness heatmap
tmp = classification_summary[classification_summary["dataset"] == "ASR"]
pivot = tmp.pivot_table(index=["regime"], columns=["target", "screening_cutoff"], values="ROC_AUC", aggfunc="mean")
fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(pivot.values, aspect="auto")
ax.set_yticks(range(len(pivot.index)), labels=[x for x in pivot.index])
ax.set_xticks(range(len(pivot.columns)), labels=[f"{a}|{b}" for a, b in pivot.columns], rotation=90)
ax.set_title("S5 Threshold robustness heatmap (ROC-AUC)", loc="left", fontweight="bold")
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
savefig("SI_threshold_robustness_heatmap")

# S6 grouped-by-metal CV details
tmp = regression_results_df[
    (regression_results_df["dataset"] == "ASR") &
    (regression_results_df["split_name"].str.contains("primary_metal", na=False))
]
fig, ax = plt.subplots(figsize=(9, 4))
for regime in tmp["regime"].unique():
    sub = tmp[tmp["regime"] == regime].groupby("model", as_index=False)["Spearman"].mean()
    ax.plot(sub["model"], sub["Spearman"], marker="o", label=regime)
ax.set_title("S6 Grouped-by-primary-metal CV details", loc="left", fontweight="bold")
ax.set_ylabel("Mean Spearman")
ax.legend(fontsize=7)
savefig("SI_grouped_by_metal_cv")

# S7 matched-pair diagnostics
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, target in zip(axes, TARGETS):
    a = matched[f"ASR__{target}"]
    b = matched[f"FSR__{target}"]
    m = a.notna() & b.notna()
    ax.scatter(a[m], b[m], s=10, alpha=0.7)
    ax.set_title(f"S7 {target}", loc="left", fontweight="bold")
    ax.set_xlabel("ASR")
    ax.set_ylabel("FSR")
savefig("SI_matched_pair_diagnostics")

# S8 coefficient / permutation importance plots
if not permutation_importance_df.empty:
    top_imp = permutation_importance_df.sort_values("importance_mean", ascending=False).head(20)
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.barh(top_imp["feature"], top_imp["importance_mean"])
    ax.set_title("S8 Top permutation importances", loc="left", fontweight="bold")
    savefig("SI_permutation_importance")

# S9 failure-case gallery table
if not failure_cases_df.empty:
    export_df(failure_cases_df.head(100), "SI_failure_case_gallery_top100", index=False)

# S10 screening-list subset analysis
if not shortlist_df.empty:
    fig, ax = plt.subplots(figsize=(8, 4))
    for target in shortlist_df["target"].unique():
        sub = shortlist_df[(shortlist_df["subset"] == "recommended_screening_only") & (shortlist_df["target"] == target)]
        if not sub.empty:
            ax.plot(sub["screening_cutoff"], sub["ROC_AUC"], marker="o", label=target)
    ax.set_xlabel("Screening cutoff")
    ax.set_ylabel("ROC-AUC")
    ax.set_title("S10 Recommended-screening subset analysis", loc="left", fontweight="bold")
    ax.legend()
    savefig("SI_screening_subset_analysis")


In [ ]:

# ============================================================
# 18. Run summary report
# ============================================================
summary_report = {
    "generated_at": datetime.now().isoformat(),
    "project_root": str(PROJECT_ROOT),
    "output_dir": str(OUT_DIR),
    "log_file": str(LOG_FILE),
    "n_asr_rows": int(len(asr_clean)),
    "n_fsr_rows": int(len(fsr_clean)),
    "n_ion_rows": int(len(ion_clean)),
    "n_matched_pairs": int(len(matched)),
    "n_regression_rows": int(len(regression_results_df)),
    "n_classification_rows": int(len(classification_results_df)),
    "n_transfer_rows": int(len(transfer_results_df)),
    "n_failure_cases": int(len(failure_cases_df)),
}
with open(OUT_DIR / "run_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary_report, f, indent=2)
log("Run summary JSON written.")
summary_report



## Notes for manuscript integration

The generated outputs are arranged so they can be used directly in manuscript drafting:

- `tables/regression_summary.csv` for headline descriptor-regime comparisons  
- `tables/classification_summary.csv` for threshold-robust screening claims  
- `tables/transfer_results.csv` for ASR↔FSR transfer-validation discussion  
- `tables/chemistry_subgroup_performance.csv` for metal / OMS / topology discussion  
- `tables/failure_cases_top50.csv` for chemistry-facing failure analysis  
- `tables/shortlist_subset_performance.csv` for practical shortlist discussion  
- `figures/Figure_1_*.png` to `Figure_5_*.png` for main text  
- `figures/SI_*.png` and the SI CSVs for supplementary information  

If you want to tighten runtime further, reduce `N_RANDOM_REPEATS`, or temporarily restrict the benchmark to one target and one dataset first.
